In [1]:
# Install and import required packages
!pip install ultralytics

import ultralytics
import cv2
from ultralytics.utils.downloads import safe_download
from ultralytics import solutions

ultralytics.checks()

Ultralytics 8.3.194  Python-3.12.7 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce GTX 1660 Ti, 6144MiB)
Setup complete  (12 CPUs, 15.9 GB RAM, 39.7/232.1 GB disk)


In [2]:
# Custom ObjectCounter class with NO region lines and BIGGER count display
import cv2
import numpy as np
from ultralytics import solutions
from ultralytics.solutions.solutions import BaseSolution, SolutionAnnotator, SolutionResults
from ultralytics.utils.plotting import colors
from collections import defaultdict
from typing import Any

class ObjectCounterNoRegion(BaseSolution):
    """
    Custom ObjectCounter class with region lines hidden and BIGGER count display
    """
    def __init__(self, **kwargs: Any) -> None:
        """Initialize the ObjectCounter class for real-time object counting in video streams."""
        super().__init__(**kwargs)

        self.in_count = 0  # Counter for objects moving inward
        self.out_count = 0  # Counter for objects moving outward
        self.counted_ids = []  # List of IDs of objects that have been counted
        self.classwise_count = defaultdict(lambda: {"IN": 0, "OUT": 0})  # Dictionary for counts, categorized by class
        self.region_initialized = False  # Flag indicating whether the region has been initialized

        self.show_in = self.CFG["show_in"]
        self.show_out = self.CFG["show_out"]
        self.margin = self.line_width * 2  # Scales the background rectangle size to display counts properly

    def count_objects(
        self,
        current_centroid: tuple[float, float],
        track_id: int,
        prev_position: tuple[float, float] | None,
        cls: int,
    ) -> None:
        """Count objects within a polygonal or linear region based on their tracks."""
        if prev_position is None or track_id in self.counted_ids:
            return

        if len(self.region) == 2:  # Linear region (defined as a line segment)
            if self.r_s.intersects(self.LineString([prev_position, current_centroid])):
                # Determine orientation of the region (vertical or horizontal)
                if abs(self.region[0][0] - self.region[1][0]) < abs(self.region[0][1] - self.region[1][1]):
                    # Vertical region: Compare x-coordinates to determine direction
                    if current_centroid[0] > prev_position[0]:  # Moving right
                        self.in_count += 1
                        self.classwise_count[self.names[cls]]["IN"] += 1
                    else:  # Moving left
                        self.out_count += 1
                        self.classwise_count[self.names[cls]]["OUT"] += 1
                # Horizontal region: Compare y-coordinates to determine direction
                elif current_centroid[1] > prev_position[1]:  # Moving downward
                    self.in_count += 1
                    self.classwise_count[self.names[cls]]["IN"] += 1
                else:  # Moving upward
                    self.out_count += 1
                    self.classwise_count[self.names[cls]]["OUT"] += 1
                self.counted_ids.append(track_id)

        elif len(self.region) > 2:  # Polygonal region
            if self.r_s.contains(self.Point(current_centroid)):
                # Determine motion direction for vertical or horizontal polygons
                region_width = max(p[0] for p in self.region) - min(p[0] for p in self.region)
                region_height = max(p[1] for p in self.region) - min(p[1] for p in self.region)

                if (
                    region_width < region_height
                    and current_centroid[0] > prev_position[0]
                    or region_width >= region_height
                    and current_centroid[1] > prev_position[1]
                ):  # Moving right or downward
                    self.in_count += 1
                    self.classwise_count[self.names[cls]]["IN"] += 1
                else:  # Moving left or upward
                    self.out_count += 1
                    self.classwise_count[self.names[cls]]["OUT"] += 1
                self.counted_ids.append(track_id)

    def display_counts(self, plot_im) -> None:
        """Display object counts on the input image or frame with BIGGER text."""
        # Use our custom big count display function
        plot_im = draw_big_count_display(plot_im, self.in_count, self.out_count)

    def process(self, im0) -> SolutionResults:
        """Process input data (frames or object tracks) and update object counts."""
        if not self.region_initialized:
            self.initialize_region()
            self.region_initialized = True

        self.extract_tracks(im0)  # Extract tracks
        self.annotator = SolutionAnnotator(im0, line_width=self.line_width)  # Initialize annotator

        # NO REGION DRAWING - This is the key difference!
        # The original code had: self.annotator.draw_region(...)
        # We completely removed it here

        # Iterate over bounding boxes, track ids and classes index
        for box, track_id, cls, conf in zip(self.boxes, self.track_ids, self.clss, self.confs):
            # Draw bounding box and counting region
            self.annotator.box_label(box, label=self.adjust_box_label(cls, conf, track_id), color=colors(cls, True))
            self.store_tracking_history(track_id, box)  # Store track history

            # Store previous position of track for object counting
            prev_position = None
            if len(self.track_history[track_id]) > 1:
                prev_position = self.track_history[track_id][-2]
            self.count_objects(self.track_history[track_id][-1], track_id, prev_position, cls)  # object counting

        plot_im = self.annotator.result()
        self.display_counts(plot_im)  # Display the counts on the frame
        self.display_output(plot_im)  # Display output with base class function

        # Return SolutionResults
        return SolutionResults(
            plot_im=plot_im,
            in_count=self.in_count,
            out_count=self.out_count,
            classwise_count=dict(self.classwise_count),
            total_tracks=len(self.track_ids),
        )

print("✅ Custom ObjectCounter class created with NO region lines and BIGGER count display!")


✅ Custom ObjectCounter class created with NO region lines and BIGGER count display!


In [3]:
# Custom function to draw readable IN/OUT/TOTAL boxes (2K-ready)
def draw_big_count_display(image, in_count, out_count):
    """
    Draw three white boxes (IN, OUT, TOTAL) at the top-right with blue, bold, uppercase text.
    Scales for 2K streams and centers text inside each box.
    """
    h, w = image.shape[:2]

    # Colors (BGR)
    bg_color = (255, 255, 255)       # white box
    txt_color = (255, 0, 0)          # blue text

    # Resolution-aware scaling (baseline 2560 width)
    base_width = 2560
    scale = max(0.9, w / base_width)

    # Font and weight
    font = cv2.FONT_HERSHEY_SIMPLEX
    font_scale = max(0.9, 0.9 * scale) * 1.2  # slightly larger for readability
    thickness = max(2, int(2 * scale))        # thicker strokes == bolder look

    # Labels (uppercase) and values; TOTAL as IN - OUT
    items = [
        ("IN", str(in_count)),
        ("OUT", str(out_count)),
        ("TOTAL", str(max(0, in_count + out_count))),
    ]

    # Spacing
    margin = int(12 * scale)               # inner padding
    gap_h = int(0.012 * h)                 # top margin
    gap_w = int(0.008 * w)                 # horizontal gap between boxes

    # Measure boxes
    specs = []
    for label, value in items:
        display_txt = f"{label}: {value}"
        (tw, th), bl = cv2.getTextSize(display_txt, font, font_scale, thickness)
        box_w = tw + margin * 2 + 8
        box_h = th + margin * 2 + bl
        specs.append((display_txt, tw, th, bl, box_w, box_h))

    # Compute total width and anchor at top-right
    total_width = sum(s[4] for s in specs) + gap_w * (len(specs) - 1)
    x_right = w - gap_w
    y_top = gap_h
    x_cursor = x_right - total_width

    # Draw boxes left->right
    for display_txt, tw, th, bl, box_w, box_h in specs:
        x1, y1 = x_cursor, y_top
        x2, y2 = x_cursor + box_w, y_top + box_h
        cv2.rectangle(image, (x1, y1), (x2, y2), bg_color, -1)
        # Center text
        tx = x1 + (box_w - tw) // 2
        ty = y1 + (box_h + th) // 2
        cv2.putText(image, display_txt, (int(tx), int(ty)), font, font_scale, txt_color, thickness, cv2.LINE_AA)
        x_cursor = x2 + gap_w

    return image

print("✅ Custom big count display function created!")


✅ Custom big count display function created!


In [4]:
# Define region coordinates for counting
# Format: [(x1, y1), (x2, y2), (x3, y3), ...] for polygon
# Format: [(x1, y1), (x2, y2)] for line
region_points = [(1205, 253), (1169, 834), (1650, 484)]  # Triangle region
# region_points = [(933, 522), (1489, 780)]  # Line region

print(f"Region points: {region_points}")


Region points: [(1205, 253), (1169, 834), (1650, 484)]


In [6]:
# Setup video capture
rtsp_url = ("rtsp://localhost:8554/live")
cap = cv2.VideoCapture(rtsp_url)
assert cap.isOpened(), "Error reading video file"

# Get video properties
w, h, fps = (int(cap.get(x)) for x in (cv2.CAP_PROP_FRAME_WIDTH,
                                       cv2.CAP_PROP_FRAME_HEIGHT,
                                       cv2.CAP_PROP_FPS))

print(f"Video properties: {w}x{h} @ {fps} FPS")

# Create output directory if it doesn't exist
import os
output_dir = r"D:\Yolo_obj_detection\output_videos"
os.makedirs(output_dir, exist_ok=True)

# Setup video writer for output (save in specific directory)
output_path = os.path.join(output_dir, "counting_output.mp4")
video_writer = cv2.VideoWriter(output_path,
                               cv2.VideoWriter_fourcc(*"mp4v"),
                               fps, (w, h))

print(f"✅ Video setup complete!")
print(f"Output will be saved to: {output_path}")


Video properties: 2560x1440 @ 30 FPS
✅ Video setup complete!
Output will be saved to: D:\Yolo_obj_detection\output_videos\counting_output.mp4


In [7]:
# Initialize ObjectCounter (NO region lines will be drawn)
counter = ObjectCounterNoRegion(
    show=True,  # Display the output
    region=region_points,  # Pass region points
    model="yolo11l.pt",  # Use smaller model for better performance
    classes=[0],  # Class 0 = person (COCO dataset)
    show_in=True,  # Display in counts
    show_out=True,  # Display out counts
    tracker="botsort.yaml",  # Tracking algorithm
    device="cuda",  # Use GPU for better performance
    line_width=1,  # Thinner lines for better performance
)

print("✅ ObjectCounter initialized with NO region lines!")
print("Using GPU and optimized settings for better performance.")
print("Count display will be BIG and visible!")


Ultralytics Solutions:  {'source': None, 'model': 'yolo11l.pt', 'classes': [0], 'show_conf': True, 'show_labels': True, 'region': [(1205, 253), (1169, 834), (1650, 484)], 'colormap': 21, 'show_in': True, 'show_out': True, 'up_angle': 145.0, 'down_angle': 90, 'kpts': [6, 8, 10], 'analytics_type': 'line', 'figsize': (12.8, 7.2), 'blur_ratio': 0.5, 'vision_point': (20, 20), 'crop_dir': 'cropped-detections', 'json_file': None, 'line_width': 1, 'records': 5, 'fps': 30.0, 'max_hist': 5, 'meter_per_pixel': 0.05, 'max_speed': 120, 'show': True, 'iou': 0.7, 'conf': 0.25, 'device': 'cuda', 'max_det': 300, 'half': False, 'tracker': 'botsort.yaml', 'verbose': True, 'data': 'images'}
✅ ObjectCounter initialized with NO region lines!
Using GPU and optimized settings for better performance.
Count display will be BIG and visible!


In [8]:
# Process video frames with performance optimizations and ERROR FIX
import time

print("Starting video processing...")
print("Press 'q' to quit the display window")

# Performance tracking
frame_count = 0
start_time = time.time()

while cap.isOpened():
    success, im0 = cap.read()
    if not success:
        print("Video processing completed!")
        break
    
    # Process frame and count objects (NO region lines will be drawn)
    results = counter(im0)
    
    # Write frame to output video
    video_writer.write(results.plot_im)
    
    # Performance tracking
    frame_count += 1
    if frame_count % 30 == 0:  # Print progress every 30 frames
        elapsed_time = time.time() - start_time
        if elapsed_time > 0:  # FIX: Prevent division by zero error
            fps_actual = frame_count / elapsed_time
            print(f"Processed {frame_count} frames at {fps_actual:.1f} FPS")

# Cleanup
cap.release()
video_writer.release()
cv2.destroyAllWindows()

# Final performance stats with ERROR FIX
total_time = time.time() - start_time
if total_time > 0:  # FIX: Prevent division by zero error
    avg_fps = frame_count / total_time
    print(f"\n✅ Processing complete!")
    print(f"Total frames processed: {frame_count}")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Average FPS: {avg_fps:.2f}")
else:
    print(f"\n✅ Processing complete!")
    print(f"Total frames processed: {frame_count}")
    print(f"Processing was very fast!")

print(f"Final counts - IN: {counter.in_count}, OUT: {counter.out_count}")
print(f"Output video saved as: {output_path}")


Starting video processing...
Press 'q' to quit the display window
WARNING no tracks found!
0: 1440x2560 91.6ms, 
Speed: 1073.8ms track, 91.6ms solution per image at shape (1, 3, 1440, 2560)

WARNING no tracks found!
1: 1440x2560 9.7ms, 
Speed: 88.7ms track, 9.7ms solution per image at shape (1, 3, 1440, 2560)

WARNING no tracks found!
2: 1440x2560 9.0ms, 
Speed: 93.1ms track, 9.0ms solution per image at shape (1, 3, 1440, 2560)

WARNING no tracks found!
3: 1440x2560 9.6ms, 
Speed: 80.9ms track, 9.6ms solution per image at shape (1, 3, 1440, 2560)

WARNING no tracks found!
4: 1440x2560 8.4ms, 
Speed: 87.4ms track, 8.4ms solution per image at shape (1, 3, 1440, 2560)

WARNING no tracks found!
5: 1440x2560 9.2ms, 
Speed: 89.7ms track, 9.2ms solution per image at shape (1, 3, 1440, 2560)

WARNING no tracks found!
6: 1440x2560 9.7ms, 
Speed: 85.1ms track, 9.7ms solution per image at shape (1, 3, 1440, 2560)

WARNING no tracks found!
7: 1440x2560 10.5ms, 
Speed: 89.6ms track, 10.5ms solution

KeyboardInterrupt: 